# Lineare SVM: Synthetischer, linear trennbarer Datensatz

In diesem Notebook wird ein **künstlicher Datensatz** erzeugt, der sich **linear trennen** lässt.
Darauf wird eine Support Vector Machine (SVM) mit linearem Kernel trainiert.

**Ablauf:**
1. Datensatz generieren (`make_blobs`)
2. Train-/Test-Split
3. SVM trainieren und bewerten
4. Supportvektoren und Trennebene plotten und als SVG exportieren

*Hinweis:* Da die Daten direkt in einem gemeinsamen Wertebereich erzeugt werden,
ist hier keine Skalierung notwendig.

## 1. Datensatz generieren

Mit `make_blobs` werden zwei kompakte Punktwolken erzeugt.
Durch den festen `random_state` ist das Ergebnis reproduzierbar,
und mit `cluster_std = 1.0` liegen die beiden Klassen klar getrennt.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs

X, y = make_blobs(
    n_samples=200,       # Anzahl der Datenpunkte
    centers=2,           # zwei Klassen
    cluster_std=1.0,     # kompakte Cluster -> linear trennbar
    center_box=(-5, 5),  # Bereich der Clusterzentren
    random_state=42      # Reproduzierbarkeit
)

print("X:", X.shape, " y:", y.shape)
print("Klassenverteilung:", np.bincount(y))

## 2. Daten visualisieren

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.set_cmap("bwr")
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", linewidths=0.3)
plt.xlabel("Merkmal 1")
plt.ylabel("Merkmal 2")
plt.title("Generierter, linear trennbarer Datensatz")
plt.show()

## 3. Train-/Test-Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0, test_size=0.25
)

## 4. SVM erstellen und trainieren

Da die Daten linear trennbar sind, findet die SVM eine Trennebene,
die beide Klassen fehlerfrei separiert -- die Testgenauigkeit sollte 1.0 betragen.
Nur wenige Punkte werden als **Supportvektoren** benötigt.

In [ ]:
from sklearn.svm import SVC

model = SVC(kernel="linear")
model.fit(X_train, y_train)

print(f"Genauigkeit auf den Testdaten: {model.score(X_test, y_test):.3f}")
print(f"Anzahl Supportvektoren: {model.support_vectors_.shape[0]}")

## 5. Supportvektoren und Trennebene

Die folgende Grafik zeigt:
- die **Trennebene** (durchgezogene Linie, Entscheidungsfunktion = 0),
- die **Margins** (gestrichelte Linien, Entscheidungsfunktion = ±1),
- die **Supportvektoren** (umkreiste Punkte).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Trainingsdaten plotten
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train,
           cmap="bwr", edgecolors="k", linewidths=0.3, zorder=2)

# Gitter über den Datenbereich aufspannen
x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

# Entscheidungsfunktion auf dem Gitter auswerten
Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Trennebene (Z = 0) und Margins (Z = -1, Z = +1) einzeichnen
ax.contour(xx, yy, Z, levels=[-1, 0, 1],
           colors="k", linestyles=["--", "-", "--"], linewidths=[1, 1.8, 1])

# Supportvektoren hervorheben
ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
           s=180, facecolors="none", edgecolors="limegreen",
           linewidths=1.8, label="Support Vector", zorder=3)

ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(loc="best")

plt.show()